中文商城评价数据集

In [1]:
from datasets import load_dataset
# 不在支持脚本加载的数据集，找一个新的
chinese_dataset = load_dataset("shijli/amazon-reviews-multi", "zh")
chinese_dataset

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})

In [2]:
def show_samples(dataset, num_samples=3, seed=40):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for item in sample:
        print(f">>> Title: {item['review_title']}")
        print(f">>> Review: {item['review_body']}")
show_samples(chinese_dataset)

>>> Title: 重修版的结局
>>> Review: 重修版的结局还是跟原版没什么很大出入...虽然说把原来伏笔用上句式古风可是结局感觉给人有点仓促
>>> Title: 盗版书！！！
>>> Review: 这是盗版书，中间翻不开，胶装在一起的，如果想要翻开看全图，书都要撕掉的感觉，装订特别烂，我才翻了几次就有掉页的趋势，后悔后悔
>>> Title: 一分钱一分货
>>> Review: 除了便宜真没什么好的，即便4档面包靠里一面还是会胡，设7档有什么意义。


In [3]:
chinese_dataset.set_format("pandas")
chinese_df = chinese_dataset["train"][:]

chinese_df["product_category"].value_counts()[:20]

product_category
book                      63058
digital_ebook_purchase    19006
apparel                   11804
shoes                      9877
beauty                     9401
kitchen                    9170
home                       8222
other                      7525
grocery                    7425
wireless                   6432
baby_product               6172
drugstore                  6072
sports                     6015
pc                         4821
toy                        3670
home_improvement           3239
watch                      3133
electronics                3059
luggage                    2984
office_product             2855
Name: count, dtype: int64

In [4]:
chinese_df.head()

,review_id,product_id,reviewer_id,stars,review_body,review_title,language,product_category
0,zh_0626061,product_zh_0691762,reviewer_zh_0824776,1,本人账号被盗，资金被江西（杨建）挪用，请亚马逊尽快查实，将本人的200元资金退回。本人已于2...,此书不是本人购买,zh,book
1,zh_0713738,product_zh_0123483,reviewer_zh_0518940,1,这简直就是太差了！出版社怎么就能出版吗？我以为是百度摘录呢！这到底是哪个鱼目混珠的教授啊？！...,简直是废话！,zh,book
2,zh_0621612,product_zh_0670618,reviewer_zh_0040023,1,购买页面显示1～2日发货，付款之后显示半个月后送达，实际收到商品距离下单日期已经一个多月。 ...,最牛逼的预售,zh,home_improvement
3,zh_0757997,product_zh_0379151,reviewer_zh_0794363,1,音箱播放时断断续续的！质量完全不行，第一次在亚马逊买东西，晕！怎么是这样的呀？有客服和我联系吗？,迷你音响,zh,other
4,zh_0086548,product_zh_0280958,reviewer_zh_0726131,1,字太小啦，建议买别的版本，慎买呀，亲们，我后悔买了这个版本！！！,排版太密，不适合菜鸟用，看到眼睛花了！,zh,book


In [5]:
def filter_books(example):
    return example['product_category'] == 'book'
chinese_dataset.reset_format()
chinese_dataset = chinese_dataset.filter(filter_books)
show_samples(chinese_dataset)

>>> Title: 非常推荐！
>>> Review: 一本从教育学角度考察慕课发展的书 对理清慕课发展历程非常有帮助！
>>> Title: 唠叨
>>> Review: 一个人的碎碎念，就是没地方唠叨了，全跑书里来喋喋不休
>>> Title: 心碎
>>> Review: 只有一层塑料袋装着，到的时候皱皱巴巴不成样子，我的地图啊，亚马逊你让我太失望了（还跟那么多书一起买的，包装都没有），心碎


In [6]:
books_dataset = chinese_dataset.filter(lambda x : len(x['review_title']) > 4)

In [7]:
books_dataset

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 36452
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 902
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 904
    })
})

In [8]:
# books_dataset["validation"] = books_dataset["validation"].select(range(20))

In [9]:
books_dataset

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 36452
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 902
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 904
    })
})

In [10]:
from transformers import AutoTokenizer
model_checkpoint = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [11]:
inputs = tokenizer("我的家在东北松花江上")
inputs

{'input_ids': [259, 35426, 3203, 1083, 107426, 12150, 4366, 6594, 1644, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
tokenizer.convert_ids_to_tokens(inputs['input_ids'])

['▁', '我的', '家', '在', '东北', '松', '花', '江', '上', '</s>']

In [13]:
max_input_length = 512
max_target_length = 30

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"], max_length=max_input_length, truncation=True
    )
    labels = tokenizer(
       text_target=examples["review_title"], max_length=max_input_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [14]:
tokenized_datasets = books_dataset.map(preprocess_function, batched=True)

In [15]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 36452
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 902
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 904
    })
})

模型评估
pip install rouge_score
https://blog.csdn.net/mch2869253130/article/details/89810974

In [16]:
import evaluate
rouge_score = evaluate.load("CZLC/rouge_raw")

In [17]:
generated_summary= tokenizer("我就瞅你咋地啊")
reference_summary= tokenizer("你瞅我没咋地")

In [18]:
scores = rouge_score.compute(
    predictions=[tokenizer.decode(generated_summary['input_ids'][1:-1])], references=[tokenizer.decode(reference_summary['input_ids'][1:-1])]
)
# scores = rouge_score.compute(
#     predictions=[generated_summary['input_ids'][1:-1]], references=[reference_summary['input_ids'][1:-1]]
# )
scores

{'1_low_precision': 0.0,
 '1_low_recall': 0.0,
 '1_low_fmeasure': 0.0,
 '1_mid_precision': 0.0,
 '1_mid_recall': 0.0,
 '1_mid_fmeasure': 0.0,
 '1_high_precision': 0.0,
 '1_high_recall': 0.0,
 '1_high_fmeasure': 0.0,
 '2_low_precision': 0.0,
 '2_low_recall': 0.0,
 '2_low_fmeasure': 0.0,
 '2_mid_precision': 0.0,
 '2_mid_recall': 0.0,
 '2_mid_fmeasure': 0.0,
 '2_high_precision': 0.0,
 '2_high_recall': 0.0,
 '2_high_fmeasure': 0.0,
 'L_low_precision': 0.0,
 'L_low_recall': 0.0,
 'L_low_fmeasure': 0.0,
 'L_mid_precision': 0.0,
 'L_mid_recall': 0.0,
 'L_mid_fmeasure': 0.0,
 'L_high_precision': 0.0,
 'L_high_recall': 0.0,
 'L_high_fmeasure': 0.0}

In [19]:
predictions = ["hello world", "general kenobi"]
references = [["hello world"], ["general kenobi"]]

# 3. 计算 ROUGE 分数
results = rouge_score.compute(predictions=predictions, references=predictions)
print(results)

{'1_low_precision': 1.0, '1_low_recall': 1.0, '1_low_fmeasure': 1.0, '1_mid_precision': 1.0, '1_mid_recall': 1.0, '1_mid_fmeasure': 1.0, '1_high_precision': 1.0, '1_high_recall': 1.0, '1_high_fmeasure': 1.0, '2_low_precision': 1.0, '2_low_recall': 1.0, '2_low_fmeasure': 1.0, '2_mid_precision': 1.0, '2_mid_recall': 1.0, '2_mid_fmeasure': 1.0, '2_high_precision': 1.0, '2_high_recall': 1.0, '2_high_fmeasure': 1.0, 'L_low_precision': 1.0, 'L_low_recall': 1.0, 'L_low_fmeasure': 1.0, 'L_mid_precision': 1.0, 'L_mid_recall': 1.0, 'L_mid_fmeasure': 1.0, 'L_high_precision': 1.0, 'L_high_recall': 1.0, 'L_high_fmeasure': 1.0}


开始建模

In [20]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.config.use_cache = False
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> MT5 number of parameters: {round(distilbert_num_parameters)}M'")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


'>>> MT5 number of parameters: 300M'


In [ ]:
from transformers import Seq2SeqTrainingArguments

batch_size = 8
eval_batch_size = 4 # 评估时的批量大小 小一点 否则显存不够用
num_train_epochs = 1
logging_steps = len(tokenized_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

args = Seq2SeqTrainingArguments(
    output_dir="./results/"+f"{model_name}-finetuned-amazon",
    # eval_strategy="epoch",
    eval_strategy="steps",  # 或 "epoch"
    eval_steps = 1000,
    # eval_accumulation_steps = 2,
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=eval_batch_size,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True, # 评估的时候需要生成的结果
    logging_steps=logging_steps,
    save_strategy="epoch",
)

In [22]:
len(tokenized_datasets["train"])

36452

In [23]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # 将 -100 替换为 pad_token_id（若无则用 0）
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    labels = np.where(labels == -100, pad_id, labels)
    predictions = np.where(predictions == -100, pad_id, predictions)
    predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # 移除 <extra_id_0> 标签
    predictions = [text.replace("<extra_id_0>", "") for text in predictions if text != ""]
    print(predictions[:2])
    results = rouge_score.compute(
        predictions=predictions, 
        references=tokenizer.batch_decode(labels, skip_special_tokens=True)
    )
    if results is None:
        return {}
    return results

In [24]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

tokenized_datasets=tokenized_datasets.remove_columns(
    books_dataset["train"].column_names
)

In [25]:
features = [tokenized_datasets["train"][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[   259,  63391,  62676,   3916,  55195,    261,  48084,   3916,  27766,
            312,  29500,  10389,    271, 208226,   2151,    261,  20256,   6751,
           6890, 168402, 200163,  18631,  14083,    261,   3661,  63391,    493,
           3464,   4074,  48084,  18558,   3917,    306,  63391,  10293,   5162,
           5624,    848,   1003,    891,   1249,    838, 150845,  18558,  31722,
          38377,    261, 128307,   2811,   4137,    848,   1322,  23971,  15327,
          50679,    291,   6751,   6890, 168402,  17077,  30733,    291,  20256,
          10428,  63391,   8149, 106714, 137270,    306,      1,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0],
        [   259,   5144,  40695,   8189,  10170,   8227,  15915,   1322,    309,
         136343,   7427, 123323,  72830,   5674,    291,   3003, 110706,   1543,
          597

In [26]:
from transformers import Seq2SeqTrainer

# 这里不能使用Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [27]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

844

In [28]:
trainer.train()

Step,Training Loss,Validation Loss,1 Low Precision,1 Low Recall,1 Low Fmeasure,1 Mid Precision,1 Mid Recall,1 Mid Fmeasure,1 High Precision,1 High Recall,1 High Fmeasure,2 Low Precision,2 Low Recall,2 Low Fmeasure,2 Mid Precision,2 Mid Recall,2 Mid Fmeasure,2 High Precision,2 High Recall,2 High Fmeasure,L Low Precision,L Low Recall,L Low Fmeasure,L Mid Precision,L Mid Recall,L Mid Fmeasure,L High Precision,L High Recall,L High Fmeasure
1000,No log,3.438319,0.099764,0.106954,0.095146,0.115948,0.124113,0.109756,0.132076,0.141707,0.125124,0.036014,0.029695,0.030370,0.047386,0.039336,0.039786,0.060238,0.051410,0.051596,0.099907,0.107118,0.094902,0.115849,0.124357,0.109873,0.131794,0.139754,0.124074
2000,No log,3.315804,0.097911,0.118780,0.099250,0.113262,0.137489,0.114118,0.128293,0.155828,0.129027,0.036791,0.036350,0.034112,0.048527,0.048269,0.045057,0.060688,0.062067,0.057214,0.097916,0.119458,0.099292,0.112768,0.137214,0.113951,0.127344,0.152999,0.127335
3000,No log,3.238519,0.094507,0.116022,0.095310,0.109493,0.134105,0.109400,0.124000,0.151970,0.123680,0.031179,0.033198,0.029898,0.042645,0.044446,0.040072,0.054559,0.058223,0.051340,0.094770,0.116435,0.095446,0.109204,0.134050,0.109053,0.124522,0.151448,0.123464
4000,No log,3.188431,0.091890,0.115281,0.093424,0.106912,0.132882,0.107688,0.121373,0.150366,0.122241,0.032044,0.034437,0.031041,0.043250,0.045839,0.041226,0.054947,0.059378,0.052696,0.092571,0.115487,0.093420,0.106639,0.132836,0.107526,0.122047,0.149447,0.121281
4557,4.368783,3.196276,0.093491,0.119409,0.095795,0.108216,0.137051,0.109660,0.122826,0.154197,0.123935,0.033597,0.035428,0.032087,0.045008,0.046718,0.042412,0.056690,0.060188,0.053789,0.093325,0.119519,0.095704,0.107678,0.136759,0.109382,0.123380,0.153909,0.123234


/home/moyin/llm-python/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['纸质不好,印刷不好,眼睛看着不舒服', '差评,差评']
['纸质不好,印刷不好,眼睛看着不舒服,准备退退退退退', '差评,差评!!!']
['纸质不好,印刷不好,眼睛看着不舒服,准备退退退退退', '差评,差评!!!']
['纸质不好,印刷不好,眼睛看着不舒服,准备退退退退退', '差评,差评!!!']
['纸质不好,印刷不好,眼睛看着不舒服,准备退退退退退', '差评,差评!!!']


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4557, training_loss=4.3688851080888815, metrics={'train_runtime': 612.8431, 'train_samples_per_second': 59.48, 'train_steps_per_second': 7.436, 'total_flos': 4000858891468800.0, 'train_loss': 4.3688851080888815, 'epoch': 1.0})

In [29]:
from transformers import pipeline
from transformers.pipelines import SUPPORTED_TASKS
for task_name, task_info in SUPPORTED_TASKS.items():
    print(f"- {task_name}")
# summarizer = pipeline("any-to-any", model="./results/mt5-small-finetuned-amazon/checkpoint-46")

- audio-classification
- automatic-speech-recognition
- text-to-audio
- feature-extraction
- text-classification
- token-classification
- table-question-answering
- document-question-answering
- fill-mask
- text-generation
- zero-shot-classification
- zero-shot-image-classification
- zero-shot-audio-classification
- image-classification
- image-feature-extraction
- image-segmentation
- image-text-to-text
- object-detection
- zero-shot-object-detection
- depth-estimation
- video-classification
- mask-generation
- keypoint-matching
- any-to-any


In [32]:

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("./results/mt5-small-finetuned-amazon/checkpoint-4557")
def print_summary(idx):
    review = books_dataset["test"][idx]["review_body"]
    title = books_dataset["test"][idx]["review_title"]
    inputs = tokenizer(review, return_tensors="pt").input_ids
    # summay = summarizer(books_dataset["test"][idx]["review_body"])
    summay = model.generate(inputs, max_new_tokens=100, do_sample=False)
    summay = tokenizer.decode(summay[0], skip_special_tokens=True)
    print(review)
    print(title)
    print(summay)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [34]:
print_summary(100)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


收到的时候简直就是一本二手书，四角压坏不说，封面厚厚的一层污垢而且被人用圆珠笔涂鸦写字。三十几块买的是这种质量，只能说很无奈。
发来被人涂鸦过的二手书，对亚马逊很失望
书质量不满意
